# Feature Engineering

For prepration of the Deepleaning based representation, I needed a stronger infrustructure due to slowness of the process on my local machine, hence I moved the code to Google Colab. 

There is `mode` variable in the first cell of the notebook, which can be set to `local` or `colab` to run the code on local machine or Google Colab respectively.

<a href="https://colab.research.google.com/github/Ajandaghian/master_thesis_polimi_private/blob/master/Analysis_notebooks/feature_engineering/1_feature_engineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


In this notebook, the aim is to reach at these:
- metadata with features at **ad segment level**
- Segment EEG data at **ad segment level** (Equivalent to bins simple averaged)
- bins:
    - TS2VEC embedding of EEG data at **ad segment level**
    - FEMBA embedding of EEG data at **ad segment level**

The model ready output will be the *Join* of any of EEG with metadata on `segment_key`.

The final data is splited based on `group k fold` at subject level, stored as train, test sets.

The final normalization shall be done pre modeling based on the selected algorithm.

# 0/ Import Libraries

In [3]:
# MODE = 'colab' #or
MODE ='local'

In [4]:
import pandas as pd
import numpy as np
import os
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import pickle
import joblib
import time


from sklearn.preprocessing import StandardScaler
from scipy.stats import pointbiserialr, chi2_contingency
from sklearn.feature_selection import mutual_info_classif
from sklearn.preprocessing import LabelEncoder

In [5]:
meta_data_2 = r"../../data/processed/ad_break_level_meta/temporal_ad_break_level_PESR_data.parquet"
bin5s_path = r"../../data/processed/eeg/separated/bin5s_obt.parquet"
segment_path = r"../../data/processed/eeg/separated/segments_obt.parquet"
keys_path = r"../../data/final_ad_break_session_keys.csv"

In [6]:
if MODE == 'colab':
  from google.colab import drive

  # Mount Google Drive
  drive.mount('/content/drive')

  head_path = '/content/drive/MyDrive/thesis/'

  meta_data_2 = head_path + r"data/processed/ad_break_level_meta/temporal_ad_break_level_PESR_data.parquet"
  bin5s_path = head_path + r"data/processed/eeg/separated/bin5s_obt.parquet"
  segment_path = head_path + r"data/processed/eeg/separated/segments_obt.parquet"
  keys_path = head_path + r"data/final_ad_break_session_keys.csv"

else:
  meta_data_2 = r"../../data/processed/ad_break_level_meta/temporal_ad_break_level_PESR_data.parquet"
  bin5s_path = r"../../data/processed/eeg/separated/bin5s_obt.parquet"
  segment_path = r"../../data/processed/eeg/separated/segments_obt.parquet"
  keys_path = r"../../data/final_ad_break_session_keys.csv"

## Load the key ad break session keys

In [ ]:
ad_break_level_keys = pd.read_csv(keys_path)
print(ad_break_level_keys.shape)
ad_break_level_keys.head()

# (2835, 1)


# -- Ad Break Level Metadata --

In [ ]:
ad_break_df = pd.read_parquet(meta_data_2)
print(ad_break_df.shape)
ad_break_df.head()

# (2938, 33)

## Feature Engineering on Ad Break Level Metadata

### 1/ Create A Unique Session Key

In [10]:
ad_break_df['ad_break_session_key'] = ad_break_df['path'] + "_" + ad_break_df['brand']

**Filtering the keys**

In [11]:
ad_break_df = ad_break_df[ad_break_df['ad_break_session_key'].isin(ad_break_level_keys['ad_break_session_key'])]

### 2/ Start time to second

In [ ]:
wad_break_df['start'] = (ad_break_df['start_ms']/1000).round()

### 3/ Seen Ad Before

filling missing values in 'seen_ad_before' with Unknown

In [13]:
warnings.filterwarnings('ignore')

ad_break_df.loc[:, 'seen_ad_before'].fillna('Unknown', inplace=True)
ad_break_df['seen_ad_before'] = ad_break_df['seen_ad_before'].astype(str)

print("Seen Ad Before value counts:")
display(ad_break_df.seen_ad_before.value_counts())

Seen Ad Before value counts:


seen_ad_before
1.0        1115
0.0        1021
Unknown     699
Name: count, dtype: int64

In [14]:
# based on the survey data, we have to see how many ads of a categoy exist in a session:
print("Ad Breaks of 'alim' category seen ad before distribution:")
print(ad_break_df[ad_break_df['category'].isin(['alim'])].pivot_table(
    index=['category', 'n_alim_ads'],
    columns='seen_ad_before',
    aggfunc='size',
    fill_value=0,
    sort=False
).reset_index().sort_values(by=['category', 'n_alim_ads']).rename(columns={0: 'not_seen', 1: 'seen'})
)
print('-'*80)
print("Ad Breaks of 'bev' category seen ad before distribution:")
print(ad_break_df[ad_break_df['category'].isin(['bev'])].pivot_table(
    index=['category', 'n_bev_ads'],
    columns='seen_ad_before',
    aggfunc='size',
    fill_value=0,
    sort=False
).reset_index().sort_values(by=['category', 'n_bev_ads']).rename(columns={0: 'not_seen', 1: 'seen'})
)

Ad Breaks of 'alim' category seen ad before distribution:
seen_ad_before category  n_alim_ads  1.0  0.0  Unknown
2                  alim           1   57  106       41
1                  alim           2  199  101       34
0                  alim           3  270  139       18
--------------------------------------------------------------------------------
Ad Breaks of 'bev' category seen ad before distribution:
seen_ad_before category  n_bev_ads  1.0  0.0  Unknown
1                   bev          1   56   95       86
0                   bev          2   64   66       64
2                   bev          3   38   77       11


#### seen_ad_before_score
Creation of seen_ad_before_score:
- '1.0' -> 1
- '0.0' -> 0
- 'Unknown'
- 0.5-> number of ads is 2 and selected yes
- 0.3 -> number of ads is 3 and selected yes

In [15]:
ad_break_df['seen_ad_before_score'] = ad_break_df['seen_ad_before']

# Assign scores based on the number of ads seen before in the same category
ad_break_df.loc[
    (ad_break_df['seen_ad_before'] == '1.0') &
    (
        (ad_break_df['n_alim_ads'] == 2) | (ad_break_df['n_bev_ads'] == 2)
    ),
    'seen_ad_before_score'
]= 0.5
ad_break_df.loc[
    (ad_break_df['seen_ad_before'] == '1.0') &
    (
        (ad_break_df['n_alim_ads'] == 3) | (ad_break_df['n_bev_ads'] == 3)
    ),
    'seen_ad_before_score'
]= 0.33

ad_break_df['seen_ad_before_score'] = ad_break_df['seen_ad_before_score'].astype(str)

ad_break_df['seen_ad_before_score'].value_counts()

seen_ad_before_score
0.0        1021
Unknown     699
0.33        524
0.5         458
1.0         133
Name: count, dtype: int64

### 4/ Survey features conversion

* Top 5 features (q_content_*) are highly correlated (0.8–0.9) → represent one latent dimension → Content Perception.

* Middle block (q_ad_* except brand influence) is moderately correlated (0.5–0.8) → second dimension → Ad Evaluation.

* q_ad_brand_influence is weakly related or inverse → acts as a separate construct → Brand Impact.


In [ ]:
# ===== Helper function to create composite features as mean z-scores =====
def make_mean_zscore(df, feature_groups):
    """
    Create composite features as mean z-scores for each feature group.

    Parameters:
    -----------
    df : pd.DataFrame
        Data containing the survey columns.
    feature_groups : dict
        Dictionary like:
        {
            'content_index': ['q_content_pleasantness', 'q_content_engagement', ...],
            'ad_quality_index': ['q_ad_quality', 'q_ad_creativity', ...]
        }

    Returns:
    --------
    df_out : pd.DataFrame
        Original dataframe with new composite z-score columns added.
    """

    df_out = df.copy()
    scaler = StandardScaler()

    for new_col, cols in feature_groups.items():
        # Scale the group
        z = scaler.fit_transform(df_out[cols])
        # Compute mean z-score
        df_out[new_col] = z.mean(axis=1)
    return df_out


feature_groups = {
    'content_index': [
        'q_content_pleasantness',
        'q_content_engagement',
        'q_content_interest',
        'q_content_quality',
        'q_content_production_quality'
    ],
    'ad_quality_index': [
        'q_ad_quality',
        'q_ad_balance',
        'q_ad_creativity',
        'q_ad_engagement'
    ],
    'brand_influence': ['q_ad_brand_influence']
}

# Apply function
ad_break_df = make_mean_zscore(ad_break_df, feature_groups)
ad_break_df.drop(columns=feature_groups['content_index'] + feature_groups['ad_quality_index'] + feature_groups['brand_influence'], inplace=True)
ad_break_df.head()

### 5/ Age Group

In [17]:
ad_break_df['age_group'] = pd.cut(
    ad_break_df['age'],
    bins=[17, 24, 34, 44, 54, 64, 100],
    labels=['18-24', '25-34', '35-44', '45-54', '55-64', '65+']
)
ad_break_df.age_group.value_counts()

age_group
45-54    623
55-64    582
25-34    505
18-24    399
35-44    370
65+      356
Name: count, dtype: int64

### 6/ Total Ad Count per session

In [18]:
if MODE == 'colab':
  path = head_path + r'data/processed/ad_break_level_meta/separated_tables/experiments_long.parquet'
else:
  path= '../../data/processed/ad_break_level_meta/separated_tables/experiments_long.parquet'

ad_break_df = ad_break_df.merge(
                        pd.read_parquet(path).groupby('path').size().rename('total_ads_in_break'),
                        on='path', how='left'
                    )

ad_break_df.total_ads_in_break.head()

0    5
1    6
2    6
3    5
4    6
Name: total_ads_in_break, dtype: int64

### 7/ ad position in the break (index)

In [19]:
ad_break_df['ad_index_in_break'] = ad_break_df['stimuli_type_order'].str.replace('ad_', '').astype(int)

#### 7.1/ is_first_ad_in_break, is_last_ad_in_break

In [20]:
ad_break_df['is_first_ad_in_break'] = (
    ad_break_df['ad_index_in_break'] == 1
).astype(int)

ad_break_df['is_last_ad_in_break'] = (
    ad_break_df['ad_index_in_break'] == ad_break_df['total_ads_in_break']
).astype(int)

### 8/ Second exposure of ad (is_repeated_exposure)

In [21]:
ad_break_df['is_repeated_exposure'] = ad_break_df \
                                            .sort_values(by=['participant_id_calculated', 'experiment_sequence', 'start']) \
                                            .groupby(['participant_id_calculated', 'brand']) \
                                            .cumcount()

fixes for '< RemovedDueToNDA >' as it's always the second exposure in second session.

In [22]:
ad_break_df.loc[
                (ad_break_df['experiment_sequence'] == 2)   &
                    (ad_break_df['brand'] == '< RemovedDueToNDA >') &
                    (ad_break_df['is_repeated_exposure'] == 0)
                , 'is_repeated_exposure'] = 1

### 9/ final cleanup and dtype adjustments

#### Drop unnecessary columns

In [23]:
drop_cols = [
    'path',  'response_id',    #id columns
    'date', 'start_ms', 'end_ms', # start is already added
    'stimuli_type_order',   #already represented by ad_index_in_break
    'subcategory', 'n_bev_ads', 'n_alim_ads', # too many null values
    'birth_year', 'age', # not needed after age_group creation
    'seen_ad_before', # not needed after seen_ad_before_score creation
    'has_ad', # redundant
]
rest_of_cols = [col for col in ad_break_df.columns if col not in drop_cols]


final_col_sorting = [
    'unaided_brand_recall', # Target

# === ID columns ===
    'ad_break_session_key', # Key for joining with EEG data
    'participant_id_calculated', # avoid data leakage in splitting

# === Ad-level features ===
    'start',
    'duration',
    'category',
    'brand',
    'ad_index_in_break',
    'is_first_ad_in_break',
    'is_last_ad_in_break',
    'is_repeated_exposure',

# === Session-level features ===
    'experiment_sequence',
    'content_watched',
    'ads_break_duration_s',
    'total_ads_in_break',
    'device',

# === Survey-based ad perception ===
    'content_index',
    'ad_quality_index',
    'brand_influence',

# === Participant-level features ===
    'seen_ad_before_score',
    'age_group',
    'gender',

# === EEG data ===
    # will be added by join
]

assert set(rest_of_cols) == set(final_col_sorting), f"The remaining columns do not match the expected set. check for {set(rest_of_cols) - set(final_col_sorting) | set(final_col_sorting) - set(rest_of_cols)}"

In [24]:
ad_break_df.drop(columns=drop_cols, inplace=True)
ad_break_df = ad_break_df[final_col_sorting]

#### Fix data types

In [ ]:
ad_break_df.head()

In [26]:
ad_break_df = ad_break_df.assign(
    category = ad_break_df['category'].astype('category'),
    brand = ad_break_df['brand'].astype('category'),
    content_watched = ad_break_df['content_watched'].astype('category'),

    device = ad_break_df['device'].astype('category'),
    gender = ad_break_df['gender'].astype('category'),

    ad_index_in_break = ad_break_df['ad_index_in_break'].astype('Int64'),
    experiment_sequence = ad_break_df['experiment_sequence'].astype('Int64'),
    ads_break_duration_s = ad_break_df['ads_break_duration_s'].astype('Int64'),
    total_ads_in_break = ad_break_df['total_ads_in_break'].astype('Int64'),

    is_first_ad_in_break = ad_break_df['is_first_ad_in_break'].astype('Int64'),
    is_last_ad_in_break = ad_break_df['is_last_ad_in_break'].astype('Int64'),
    is_repeated_exposure = ad_break_df['is_repeated_exposure'].astype('Int64'),

    seen_ad_before_score = ad_break_df['seen_ad_before_score'].astype('category'),
)


print(ad_break_df.dtypes)

unaided_brand_recall            Int64
ad_break_session_key           object
participant_id_calculated       int64
start                         float64
duration                      float64
category                     category
brand                        category
ad_index_in_break               Int64
is_first_ad_in_break            Int64
is_last_ad_in_break             Int64
is_repeated_exposure            Int64
experiment_sequence             Int64
content_watched              category
ads_break_duration_s            Int64
total_ads_in_break              Int64
device                       category
content_index                 float64
ad_quality_index              float64
brand_influence               float64
seen_ad_before_score         category
age_group                    category
gender                       category
dtype: object


In [27]:
ad_break_df.dtypes[ad_break_df.dtypes.isin([int, 'Int64', float, 'float64']) == True]

Series([], dtype: object)

## Train/Test Split

We'll split the data based on groups at subject level.

```
Data:
    ├── Train (fixed, saved) -> Group wise split
    │
    └── Test (untouched)
```

In [28]:
from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(
        n_splits=1,
        test_size=0.3,
        random_state=22122025)

train_idx, test_idx = next(
    gss.split(
        ad_break_df.iloc[:, 3:],
        ad_break_df['unaided_brand_recall'],
        groups=ad_break_df['participant_id_calculated'])
)

print("Train idx:", train_idx.shape)
print("Test idx:", test_idx.shape)

# print("Train groups:", set(ad_break_df['participant_id_calculated'].iloc[train_idx]))
# print("Test groups:", set(ad_break_df['participant_id_calculated'].iloc[test_idx]))

train_df = ad_break_df.iloc[train_idx].reset_index(drop=True)
test_df = ad_break_df.iloc[test_idx].reset_index(drop=True)


Train idx: (1970,)
Test idx: (865,)


In [29]:
print(f"--- Sanity Check ---\n")

assert train_df['participant_id_calculated'].isin(test_df['participant_id_calculated']).sum() == 0, "Data leakage detected: Some participants are present in both train and test sets."
print("✅ No participant leakage detected")

--- Sanity Check ---

✅ No participant leakage detected


In [30]:
print(f"--- Balance Check ---\n")
print('ad break df shape', ad_break_df.shape)
print('train df shape', train_df.shape, round(train_df.shape[0]/ad_break_df.shape[0], 2), '% of data')
print('test df shape', test_df.shape, round(test_df.shape[0]/ad_break_df.shape[0], 2), '% of data')

print('\n' + '-'*50)

print("Overall unaided_brand_recall distribution:")
print(pd.DataFrame({
        'train': train_df.unaided_brand_recall.value_counts(normalize=False).round(2),
        'train_%': train_df.unaided_brand_recall.value_counts(normalize=True).round(2),
        'test': test_df.unaided_brand_recall.value_counts(normalize=False).round(2),
        'test_%': test_df.unaided_brand_recall.value_counts(normalize=True).round(2)
    })
    )
print('\n' + '-'*50)
print("brand-wise unaided_brand_recall distribution:(% recalled)")
print(pd.DataFrame({
        'train_recalled': pd.crosstab(train_df['brand'],train_df['unaided_brand_recall'],normalize='columns').round(2)[1].multiply(100),
        'test_recalled': pd.crosstab(test_df['brand'],test_df['unaided_brand_recall'],normalize='columns').round(2)[1].multiply(100)
    })
    )

print('\n' + '-'*50)
print("experiment_sequence-wise unaided_brand_recall distribution:(% recalled)")
print(pd.DataFrame({
        'train': train_df.experiment_sequence.value_counts(normalize=False).round(2),
        'train_%': train_df.experiment_sequence.value_counts(normalize=True).round(2),
        'test': test_df.experiment_sequence.value_counts(normalize=False).round(2),
        'test_%': test_df.experiment_sequence.value_counts(normalize=True).round(2)
    })
    )


--- Balance Check ---

ad break df shape (2835, 22)
train df shape (1970, 22) 0.69 % of data
test df shape (865, 22) 0.31 % of data

--------------------------------------------------
Overall unaided_brand_recall distribution:
                      train  train_%  test  test_%
unaided_brand_recall                              
1                      1354     0.69   541    0.63
0                       616     0.31   324    0.37

--------------------------------------------------
brand-wise unaided_brand_recall distribution:(% recalled)
                     train_recalled  test_recalled
brand                                             
< RemovedDueToNDA >                      7.0            6.0
< RemovedDueToNDA >                      6.0            5.0
< RemovedDueToNDA >                      9.0            7.0
< RemovedDueToNDA >                      7.0            5.0
< RemovedDueToNDA >                      7.0            8.0
< RemovedDueToNDA >                      10.0           1

## Scaling Numerical Features

#### Features that **should NOT be scaled**

**❌ Target variable**

* `unaided_brand_recall`
  This is the target (0/1). **Never scale the target.**

---

**❌ Binary indicators (0/1)**

* `is_first_ad_in_break`
* `is_last_ad_in_break`
* `is_repeated_exposure`

These are already meaningful as-is.
Scaling would **destroy their interpretation**.

---

**❌ Ordinal / count features**

* `ad_index_in_break` -> 1, 2, 3, ...
* `experiment_sequence` -> 1, 2
* `total_ads_in_break` -> 2, 3, 4, ...



These represent **order or counts**, not continuous magnitudes.
Scaling is unnecessary and often harmful.

**❌ Survey index features**
they are already normalized scores between 0 and 1:

* `content_index`
* `ad_quality_index`
* `brand_influence`

---

#### Features that **SHOULD be scaled**

**✅ Continuous numerical features**

These have different ranges and benefit from scaling:
* `start`

**✅ Categorical with big value**
* `ads_break_duration_s` -> 60, 120, 180 ==> large values that can skew models. I change them to 1, 2, 3
* `duration` -> 15, 30    ==> large values that can skew models. I change them to 1, 2


In [31]:
from sklearn.preprocessing import StandardScaler

scale_cols = [
    'start',
]

scaler = StandardScaler()

train_df[scale_cols] = scaler.fit_transform(train_df[scale_cols])
test_df[scale_cols]  = scaler.transform(test_df[scale_cols])

In [32]:
train_df['ads_break_duration_s'] = train_df['ads_break_duration_s'].replace({60: 1, 120: 2, 180: 3})
test_df['ads_break_duration_s']  = test_df['ads_break_duration_s'].replace({60: 1, 120: 2, 180: 3})

train_df['duration'] = train_df['duration'].replace({15: 1, 30: 2})
test_df['duration']  = test_df['duration'].replace({15: 1, 30: 2})

## Export Metadata (model ready)

In [33]:
# save the splits
if MODE == 'colab':
    train_path = head_path + r'data/final_for_modeling/train_ad_break_level_data.parquet'
    test_path = head_path + r'data/final_for_modeling/test_ad_break_level_data.parquet'
    os.makedirs(os.path.dirname(train_path), exist_ok=True)
    os.makedirs(os.path.dirname(test_path), exist_ok=True)

else:
    train_path = r'../../data/final_for_modeling/train_ad_break_level_data.parquet'
    test_path = r'../../data/final_for_modeling/test_ad_break_level_data.parquet'
    os.makedirs(os.path.dirname(train_path), exist_ok=True)
    os.makedirs(os.path.dirname(test_path), exist_ok=True)

train_df.to_parquet(train_path, index=False)
test_df.to_parquet(test_path, index=False)

# -- Segment ----------------------------

In [35]:
from scipy.stats import ttest_ind
def numeric_vs_binary(df, numeric_cols, target, alpha=0.05):
    results = []

    for col in numeric_cols:
        group0 = df[df[target] == 0][col].dropna()
        group1 = df[df[target] == 1][col].dropna()

        t_stat, p_value = ttest_ind(group0, group1, equal_var=False)

        results.append({
            "feature": col,
            "t_stat": t_stat,
            "p_value": p_value,
            "interpration": '✅' if p_value < alpha else '❌',
            "mean_0": group0.mean().round(2),
            "mean_1": group1.mean().round(2),
            "diff": (group1.mean() - group0.mean()).round(2)
        })

    return pd.DataFrame(results).sort_values(by="p_value")

In [ ]:
segment = pd.read_parquet(segment_path)
segment.head()

## Feat. Eng. on Segment

### 1/ Create A Unique Session Key

In [39]:
segment['ad_break_session_key'] = segment['key'] + "_" + segment['stimuli_name_clean']

**Filtering the keys**

In [40]:
segment = segment[segment['ad_break_session_key'].isin(ad_break_level_keys['ad_break_session_key'])]

### 2/ Feature Selection

In [ ]:
all_transformations = ["raw", "z-all", "z-program", "z-all-log", "z-program-log"]
all_cols = segment.columns.tolist()

for c in [ 'key', 'ad_break_session_key' ,'source_file','start_ms','end_ms','duration','stimuli_name_clean','stimuli_type','stimuli_type_order']:
    all_cols.remove(c)


z_all_cols = [col for col in all_cols if 'z-a_' in col]
z_program_cols = [col for col in all_cols if 'z-p_' in col]
z_all_log_cols = [col for col in all_cols if 'z-a-log_' in col]
z_program_log_cols = [col for col in all_cols if 'z-p-log_' in col]
raw_cols = [col for col in all_cols if all(x not in col for x in ['z-a_', 'z-p_', 'z-a-log_', 'z-p-log_'])]

print(
    "total number of features:", len(all_cols),
    "\nNumber of raw features:", len(raw_cols),
    "\nNumber of z-all features:", len(z_all_cols),
    "\nNumber of z-program features:", len(z_program_cols),
    "\nNumber of z-all-log features:", len(z_all_log_cols),
    "\nNumber of z-program-log features:", len(z_program_log_cols),
    )

print('='*80)

# Exclude col with electrods
all_electrodes = ["F7", "AF7", "Fp1", "Fpz", "Fp2", "AF8", "F8"]

all_montage_cols = [c for c in all_cols if not any(electrode in c for electrode in all_electrodes)]


z_all_montage_cols = [col for col in all_montage_cols if 'z-a_' in col]
z_program_montage_cols = [col for col in all_montage_cols if 'z-p_' in col]
z_all_log_montage_cols = [col for col in all_montage_cols if 'z-a-log_' in col]
z_program_log_montage_cols = [col for col in all_montage_cols if 'z-p-log_' in col]
raw_montage_cols = [col for col in all_montage_cols if all(x not in col for x in ['z-a_', 'z-p_', 'z-a-log_', 'z-p-log_'])]

print(
    "total number of montage features:", len(all_montage_cols),
    "\nNumber of raw features:", len(raw_montage_cols),
    "\nNumber of z-all features:", len(z_all_montage_cols),
    "\nNumber of z-program features:", len(z_program_montage_cols),
    "\nNumber of z-all-log features:", len(z_all_log_montage_cols),
    "\nNumber of z-program-log features:", len(z_program_log_montage_cols),
    )

total number of features: 2109 
Number of raw features: 441 
Number of z-all features: 417 
Number of z-program features: 417 
Number of z-all-log features: 417 
Number of z-program-log features: 417
total number of montage features: 1304 
Number of raw features: 280 
Number of z-all features: 256 
Number of z-program features: 256 
Number of z-all-log features: 256 
Number of z-program-log features: 256


In [42]:
temp = segment.merge(
    ad_break_df[['ad_break_session_key', 'unaided_brand_recall']],
    on='ad_break_session_key',
    how='left'
).dropna(subset=['unaided_brand_recall'])

- ttest on 'mean all' for selecting the transformation
- focusing on montage features only, because they are more generalizable and less noisy than channel-based features.
- running t-test and select the best calculation variations
- double check the t-test

ttest

In [43]:
# comparing the count of signifant ttest in
print("raw features:")
print(
    numeric_vs_binary(
        df=temp,
        numeric_cols=raw_cols,
        target='unaided_brand_recall',
    )['interpration'].value_counts(normalize=True).round(3).rename('raw')
)
print("-"*40)
print("z-all features:")
print(
    numeric_vs_binary(
        df=temp,
        numeric_cols=z_all_cols,
        target='unaided_brand_recall',
    )['interpration'].value_counts(normalize=True).round(3).rename('z-all')
)
print("-"*40)
print("z-program features:")
print(
    numeric_vs_binary(
        df=temp,
        numeric_cols=z_program_cols,
        target='unaided_brand_recall',
    )['interpration'].value_counts(normalize=True).round(3).rename('z-program')
)
print("-"*40)
print("z-all-log features:")
print(
    numeric_vs_binary(
        df=temp,
        numeric_cols=z_all_log_cols,
        target='unaided_brand_recall',
    )['interpration'].value_counts(normalize=True).round(3).rename('z-all-log')
)
print("-"*40)
print("z-program-log features:")
print(
    numeric_vs_binary(
        df=temp,
        numeric_cols=z_program_log_cols,
        target='unaided_brand_recall',
    )['interpration'].value_counts(normalize=True).round(3).rename('z-program-log')
)

raw features:
interpration
❌    0.637
✅    0.363
Name: raw, dtype: float64
----------------------------------------
z-all features:
interpration
❌    0.715
✅    0.285
Name: z-all, dtype: float64
----------------------------------------
z-program features:
interpration
❌    0.911
✅    0.089
Name: z-program, dtype: float64
----------------------------------------
z-all-log features:
interpration
❌    0.722
✅    0.278
Name: z-all-log, dtype: float64
----------------------------------------
z-program-log features:
interpration
❌    0.856
✅    0.144
Name: z-program-log, dtype: float64


In [44]:
raw_mean_all_cols = [col for col in raw_cols if '_All' in col]
raw_mean_all_cols

numeric_vs_binary(
        df=temp,
        numeric_cols=raw_mean_all_cols,
        target='unaided_brand_recall',
    ).sort_values(by=['feature', 'p_value']).round(4)

,feature,t_stat,p_value,interpration,mean_0,mean_1,diff
30,AI-1a_All,-0.0007,0.9994,❌,6.000000e+00,6.000000e+00,0.000000e+00
31,AI-1b_All,-0.0817,0.9349,❌,6.800000e+00,6.810000e+00,1.000000e-02
18,EI-1a_All,-2.2378,0.0253,✅,1.600000e-01,1.800000e-01,1.000000e-02
19,EI-1b_All,-2.0075,0.0448,✅,1.400000e-01,1.500000e-01,1.000000e-02
20,EI-1c_All,-2.2449,0.0249,✅,1.500000e-01,1.600000e-01,1.000000e-02
21,EI-1d_All,-2.0524,0.0403,✅,1.600000e-01,1.700000e-01,1.000000e-02
22,EI-2a_All,-2.9706,0.0030,✅,5.700000e-01,6.100000e-01,5.000000e-02
23,EI-2b_All,-2.3419,0.0193,✅,3.700000e-01,4.000000e-01,3.000000e-02
24,EI-2c_All,-3.0646,0.0022,✅,5.100000e-01,5.600000e-01,5.000000e-02
25,EI-2d_All,-2.9964,0.0028,✅,6.600000e-01,7.100000e-01,5.000000e-02


Result:
- attention index 1a
- engagement index EI-2c_All

- --GFP_Alpha-upper_All don't add
- -- theta lower
- -- gfp beta
- -- gfp gamma

- mean-psd_Alpha-upper _All_v^2/Hz
- beta
- gamma
- delta
- theta

In [45]:
chosen_flavor = ['AI-1a_',
                'EI-2c_',
                'mean-psd_Alpha-upper_',
                'mean-psd_Beta_',
                'mean-psd_Delta_',
                'mean-psd_Theta_',
                'mean-psd_Gamma_',]

clusters_cols = [col for col in raw_montage_cols if ('All' not in col)
                                                    and (any(col.startswith(flavor) for flavor in chosen_flavor))
                ]

print(sorted(clusters_cols))
len(clusters_cols)

['AI-1a_anterior-frontal', 'AI-1a_frontal-midline', 'AI-1a_frontolateral', 'AI-1a_frontolateral-ext', 'AI-1a_left-frontolateral-ext', 'AI-1a_prefrontal', 'AI-1a_right-frontolateral-ext', 'EI-2c_anterior-frontal', 'EI-2c_frontal-midline', 'EI-2c_frontolateral', 'EI-2c_frontolateral-ext', 'EI-2c_left-frontolateral-ext', 'EI-2c_prefrontal', 'EI-2c_right-frontolateral-ext', 'mean-psd_Alpha-upper_anterior-frontal_v^2/Hz', 'mean-psd_Alpha-upper_frontal-midline_v^2/Hz', 'mean-psd_Alpha-upper_frontolateral-ext_v^2/Hz', 'mean-psd_Alpha-upper_frontolateral_v^2/Hz', 'mean-psd_Alpha-upper_left-frontolateral-ext_v^2/Hz', 'mean-psd_Alpha-upper_prefrontal_v^2/Hz', 'mean-psd_Alpha-upper_right-frontolateral-ext_v^2/Hz', 'mean-psd_Beta_anterior-frontal_v^2/Hz', 'mean-psd_Beta_frontal-midline_v^2/Hz', 'mean-psd_Beta_frontolateral-ext_v^2/Hz', 'mean-psd_Beta_frontolateral_v^2/Hz', 'mean-psd_Beta_left-frontolateral-ext_v^2/Hz', 'mean-psd_Beta_prefrontal_v^2/Hz', 'mean-psd_Beta_right-frontolateral-ext_v^2/H

49

In [46]:
numeric_vs_binary(
        df=temp,
        numeric_cols=clusters_cols,
        target='unaided_brand_recall',
    ).sort_values(by=['feature', 'p_value']).round(4)

,feature,t_stat,p_value,interpration,mean_0,mean_1,diff
40,AI-1a_anterior-frontal,0.0430,0.9657,❌,6.01,5.99,-0.02
36,AI-1a_frontal-midline,0.4333,0.6649,❌,5.74,5.66,-0.08
42,AI-1a_frontolateral,-0.8724,0.3831,❌,7.38,7.55,0.18
44,AI-1a_frontolateral-ext,-0.1705,0.8646,❌,6.27,6.30,0.03
46,AI-1a_left-frontolateral-ext,-0.9430,0.3458,❌,6.22,6.39,0.16
38,AI-1a_prefrontal,0.5119,0.6088,❌,5.94,5.86,-0.08
48,AI-1a_right-frontolateral-ext,0.4564,0.6481,❌,6.71,6.61,-0.10
39,EI-2c_anterior-frontal,-3.2049,0.0014,✅,0.59,0.66,0.07
35,EI-2c_frontal-midline,-3.4075,0.0007,✅,0.53,0.59,0.06
41,EI-2c_frontolateral,-0.8615,0.3891,❌,0.48,0.50,0.01


In [ ]:
corr_matrix = temp[sorted(clusters_cols, reverse=True)].corr().round(2)
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

plt.figure(figsize=(20,16))
sns.heatmap(corr_matrix, mask=mask, cmap='coolwarm', vmin=-1, vmax=1, annot=True, fmt='.2f', annot_kws={"size": 6})
plt.xticks(fontsize=6)
plt.yticks(fontsize=6)
plt.show()

### 3/ final feature sorting and dtype adjustments

In [48]:
final_features = ['ad_break_session_key']
for c in clusters_cols:
    final_features.append('z-a_' + c) # using z-all transformation
print(final_features)

final_segment = segment[final_features].copy()

['ad_break_session_key', 'z-a_mean-psd_Delta_frontal-midline_v^2/Hz', 'z-a_mean-psd_Delta_prefrontal_v^2/Hz', 'z-a_mean-psd_Delta_anterior-frontal_v^2/Hz', 'z-a_mean-psd_Delta_frontolateral_v^2/Hz', 'z-a_mean-psd_Delta_frontolateral-ext_v^2/Hz', 'z-a_mean-psd_Delta_left-frontolateral-ext_v^2/Hz', 'z-a_mean-psd_Delta_right-frontolateral-ext_v^2/Hz', 'z-a_mean-psd_Theta_frontal-midline_v^2/Hz', 'z-a_mean-psd_Theta_prefrontal_v^2/Hz', 'z-a_mean-psd_Theta_anterior-frontal_v^2/Hz', 'z-a_mean-psd_Theta_frontolateral_v^2/Hz', 'z-a_mean-psd_Theta_frontolateral-ext_v^2/Hz', 'z-a_mean-psd_Theta_left-frontolateral-ext_v^2/Hz', 'z-a_mean-psd_Theta_right-frontolateral-ext_v^2/Hz', 'z-a_mean-psd_Alpha-upper_frontal-midline_v^2/Hz', 'z-a_mean-psd_Alpha-upper_prefrontal_v^2/Hz', 'z-a_mean-psd_Alpha-upper_anterior-frontal_v^2/Hz', 'z-a_mean-psd_Alpha-upper_frontolateral_v^2/Hz', 'z-a_mean-psd_Alpha-upper_frontolateral-ext_v^2/Hz', 'z-a_mean-psd_Alpha-upper_left-frontolateral-ext_v^2/Hz', 'z-a_mean-psd_

Checking the dtypes

In [49]:
# print(final_segment.dtypes)
# all good

## Export Segment

In [52]:
# save the splits
if MODE == 'colab':
    segment_path = head_path + r'data/final_for_modeling/segment_df.parquet'
else:
    segment_path = r'../../data/final_for_modeling/tsegment_df.parquet'

final_segment.to_parquet(segment_path, index=False)

# -- bins ----------------------------

In [50]:
bin = pd.read_parquet(bin5s_path)
bin.head()

,source_file,start_ms,end_ms,bin_param_duration,duration,stimuli_name_clean,stimuli_type,stimuli_type_order,bin_order,ad_order,...,z-a-log_EI-2b_right-frontolateral-ext,z-a-log_EI-2c_right-frontolateral-ext,z-a-log_EI-2d_right-frontolateral-ext,z-a-log_EI-3a_right-frontolateral-ext,z-a-log_EI-3b_right-frontolateral-ext,z-a-log_EI-3c_right-frontolateral-ext,z-a-log_EI-3d_right-frontolateral-ext,z-a-log_AI-1a_right-frontolateral-ext,z-a-log_AI-1b_right-frontolateral-ext,key
0,D:\< RemovedDueToNDA >_AssoTV\sensor-data\O1_raw_20240607\< RemovedDueToNDA >_...,805.8601,5805.8601,5000.0,5000.0,doc,program,program_1,1,<NA>,...,0.107483,-0.146553,-2.872517,-3.191236,-2.251281,-2.214035,-5.442295,-1.349950,-1.740690,E:\AssoTV\sensor-data\O1_raw_20240607\< RemovedDueToNDA >_line...
1,D:\< RemovedDueToNDA >_AssoTV\sensor-data\O1_raw_20240607\< RemovedDueToNDA >_...,5805.8601,10805.8601,5000.0,5000.0,doc,program,program_1,2,<NA>,...,3.119100,1.799718,-0.277707,-0.237048,2.040002,0.988895,-1.510551,-1.908700,-1.967045,E:\AssoTV\sensor-data\O1_raw_20240607\< RemovedDueToNDA >_line...
2,D:\< RemovedDueToNDA >_AssoTV\sensor-data\O1_raw_20240607\< RemovedDueToNDA >_...,10805.8601,15805.8601,5000.0,5000.0,doc,program,program_1,3,<NA>,...,1.757433,3.565282,1.277831,1.359586,0.732951,2.834359,-0.166243,0.572540,0.776246,E:\AssoTV\sensor-data\O1_raw_20240607\< RemovedDueToNDA >_line...
3,D:\< RemovedDueToNDA >_AssoTV\sensor-data\O1_raw_20240607\< RemovedDueToNDA >_...,15805.8601,20805.8601,5000.0,5000.0,doc,program,program_1,4,<NA>,...,2.779312,1.841002,2.560159,1.596225,2.158188,1.424904,1.469279,-3.364040,-3.189203,E:\AssoTV\sensor-data\O1_raw_20240607\< RemovedDueToNDA >_line...
4,D:\< RemovedDueToNDA >_AssoTV\sensor-data\O1_raw_20240607\< RemovedDueToNDA >_...,20805.8601,25805.8601,5000.0,5000.0,doc,program,program_1,5,<NA>,...,1.038946,1.840102,0.134628,1.483745,1.163691,2.037841,0.307296,0.074604,0.097508,E:\AssoTV\sensor-data\O1_raw_20240607\< RemovedDueToNDA >_line...


In [51]:
bin = bin[bin['stimuli_type'] == 'ad'].copy()

## Feat. Eng. on Bins

### 1/ Create A Unique Session Key

In [52]:
bin['ad_break_session_key'] = bin['key'] + "_" + bin['stimuli_name_clean']

**Filtering the keys**


In [53]:
assert bin[bin['ad_break_session_key'].isin(ad_break_level_keys['ad_break_session_key'])].groupby('ad_break_session_key').size().shape[0] == len(ad_break_level_keys)

In [54]:
bin = bin[bin['ad_break_session_key'].isin(ad_break_level_keys['ad_break_session_key'])]

### 2/ final feature sorting and dtype adjustments

In [ ]:
final_features_bin = ['ad_break_session_key', 'bin_order'] + final_features[1:]
final_bin = bin[final_features_bin].sort_values(by=['ad_break_session_key', 'bin_order']).copy()
final_bin.head()

In [56]:
print(final_bin.shape)
print(final_bin.columns)

(15607, 51)
Index(['ad_break_session_key', 'bin_order',
       'z-a_mean-psd_Delta_frontal-midline_v^2/Hz',
       'z-a_mean-psd_Delta_prefrontal_v^2/Hz',
       'z-a_mean-psd_Delta_anterior-frontal_v^2/Hz',
       'z-a_mean-psd_Delta_frontolateral_v^2/Hz',
       'z-a_mean-psd_Delta_frontolateral-ext_v^2/Hz',
       'z-a_mean-psd_Delta_left-frontolateral-ext_v^2/Hz',
       'z-a_mean-psd_Delta_right-frontolateral-ext_v^2/Hz',
       'z-a_mean-psd_Theta_frontal-midline_v^2/Hz',
       'z-a_mean-psd_Theta_prefrontal_v^2/Hz',
       'z-a_mean-psd_Theta_anterior-frontal_v^2/Hz',
       'z-a_mean-psd_Theta_frontolateral_v^2/Hz',
       'z-a_mean-psd_Theta_frontolateral-ext_v^2/Hz',
       'z-a_mean-psd_Theta_left-frontolateral-ext_v^2/Hz',
       'z-a_mean-psd_Theta_right-frontolateral-ext_v^2/Hz',
       'z-a_mean-psd_Alpha-upper_frontal-midline_v^2/Hz',
       'z-a_mean-psd_Alpha-upper_prefrontal_v^2/Hz',
       'z-a_mean-psd_Alpha-upper_anterior-frontal_v^2/Hz',
       'z-a_mean-psd_Alp

### 3/ Embeddings of Bins
to get EEG representation at ad segment level

In [60]:
# temprorary saving the necesasry data for sampling of the training time
os.makedirs(r'../../data/final_for_modeling/temp/', exist_ok=True)

joblib.dump(final_bin, r'../../data/final_for_modeling/temp/final_bin.pkl')
joblib.dump(final_features, r'../../data/final_for_modeling/temp/final_features.pkl')

['../../data/final_for_modeling/temp/final_features.pkl']

#### 3.1/ TS2VEC Embedding

raw bins
→ group by ad_break
→ build sequences (T × F)
→ train embedding model (TS2Vec)
→ pool over time
→ get 1 row per ad_break


In [60]:
# Define feature columns (excluding ID columns)
feature_cols = [
    c for c in final_bin.columns
    if c not in ["ad_break_session_key", "bin_order"]
]
print(f"Number of features: {len(feature_cols)}")

#num_series × time_steps × num_features
sequences = []
keys = []

for key, g in final_bin.groupby("ad_break_session_key"):
    g = g.sort_values("bin_order")
    X = g[feature_cols].values  # shape: (T, F)
    sequences.append(X)
    keys.append(key)

len(sequences)

Number of features: 49


2835

In [61]:
assert sequences[0].shape[1] == 49  # (time_steps, num_features)

In [62]:
if MODE == 'colab':
  %pip install ts2vec

In [63]:
from ts2vec import TS2Vec
import torch

# Pad sequences to same length and convert to 3D array
max_len = max(seq.shape[0] for seq in sequences)
padded_sequences = np.array([
    np.pad(seq, ((0, max_len - seq.shape[0]), (0, 0)), mode='constant')
    for seq in sequences
])

model = TS2Vec(
    input_dims=padded_sequences.shape[2],  # 49 for now
    output_dims=128,
    hidden_dims=64,
    depth=10,
    device= 'cuda' if torch.cuda.is_available() else 'cpu',
)

#==================== Timing the training process ====================
start = time.perf_counter()
print(">> model fitting starts <<")
model.fit(
    padded_sequences,  # a 3D numpy array: (num_series, time_steps, features)
    n_epochs=30
)
print(">> model fitting finished <<")
#=====================================================================
fit_time = time.perf_counter() - start


# ===================== Timing the embedding process ====================
start = time.perf_counter()
print(">> embedding starts <<")
embeddings = model.encode(padded_sequences)
# =====================================================================
print(">> embedding finished <<")
encode_time = time.perf_counter() - start
print("embeddings shape:", embeddings.shape)  # (num_series, time_steps, embedding_dim


final_embeddings = [
    emb.mean(axis=0)
    for emb in embeddings
]

# time report
print("=" * 40)
print("TS2Vec Feature Extraction Time Report")
print(f"TS2Vec training time: {fit_time:.2f} seconds")
print(f"TS2Vec encoding time: {encode_time:.2f} seconds")

>> model fitting starts <<
>> model fitting finished <<
>> embedding starts <<
>> embedding finished <<
embeddings shape: (2835, 7, 128)
TS2Vec Feature Extraction Time Report
TS2Vec training time: 453.70 seconds
TS2Vec encoding time: 1.82 seconds


“Zero-padding was applied to equalize sequence lengths. Since all features were z-normalized, padded values correspond to neutral feature levels, minimizing their influence on the learned representations.”

"The computational cost of the embedding stage was evaluated by measuring wall-clock execution time for both training and encoding phases using high-resolution timers."

In [64]:
bin_ts2vec_emb_df = pd.DataFrame(
    final_embeddings,
    columns=[f"ts2vec_{i}" for i in range(embeddings.shape[2])]
)

bin_ts2vec_emb_df["ad_break_session_key"] = keys


bin_ts2vec_emb_df.head()


,ts2vec_0,ts2vec_1,ts2vec_2,ts2vec_3,ts2vec_4,ts2vec_5,ts2vec_6,ts2vec_7,ts2vec_8,ts2vec_9,...,ts2vec_119,ts2vec_120,ts2vec_121,ts2vec_122,ts2vec_123,ts2vec_124,ts2vec_125,ts2vec_126,ts2vec_127,ad_break_session_key
0,0.042692,0.159486,-0.118078,-0.057548,-0.288435,-0.117112,0.134354,-0.108398,0.231327,-0.172800,...,-0.017132,0.158900,0.144564,-0.068481,-0.178288,-0.407387,-0.166404,-0.014168,-0.264605,E:\AssoTV\sensor-data\O1_raw_20240607\< RemovedDueToNDA >_line...
1,-0.135814,0.222624,0.115961,0.049484,-0.115078,-0.104692,0.073016,-0.241724,0.255893,-0.225113,...,-0.206212,0.058676,0.136310,-0.213397,-0.112412,-0.324680,-0.337670,-0.112463,-0.190669,E:\AssoTV\sensor-data\O1_raw_20240607\< RemovedDueToNDA >_line...
2,-0.082658,0.282650,-0.079302,-0.072278,-0.127344,-0.049781,0.077997,-0.032921,0.221576,-0.285925,...,-0.121943,-0.029885,0.323554,-0.252236,-0.107037,-0.295085,-0.240081,-0.176698,-0.175266,E:\AssoTV\sensor-data\O1_raw_20240607\< RemovedDueToNDA >_line...
3,-0.067795,0.123047,-0.160287,0.020815,-0.282471,-0.245127,0.027181,0.137064,0.177798,-0.198899,...,-0.122814,0.046004,0.233662,-0.195752,-0.188890,-0.621293,0.006251,-0.036134,-0.314927,E:\AssoTV\sensor-data\O1_raw_20240607\< RemovedDueToNDA >_line...
4,-0.127902,-0.251118,-0.129263,-0.257251,-0.334099,-0.249328,0.031411,-0.192752,-0.013076,-0.276291,...,-0.343321,0.052170,-0.208035,-0.066993,-0.171670,-0.366352,-0.188573,-0.253447,-0.428756,E:\AssoTV\sensor-data\O1_raw_20240607\< RemovedDueToNDA >_line...


In [65]:
# Check for missing values
bin_ts2vec_emb_df.isna().any().sum()

np.int64(0)

#### Export TS2VEC Embedding DataFrame

In [66]:
bin_ts2vec_emb_df.shape

(2835, 129)

In [67]:
# save the splits
if MODE == 'colab':
    ts2vec_path = head_path + r'data/final_for_modeling/bin_ts2vec_emb_df.parquet'

else:
    ts2vec_path = r'../../data/final_for_modeling/bin_ts2vec_emb_df.parquet'

bin_ts2vec_emb_df.to_parquet(ts2vec_path, index=False)

#### 3.2/ FEMBA Embedding

FEMBA = Frequency-Enhanced Multivariate time-series Embedding via Bottleneck Autoencoder

Input sequence
   ->
Frequency-aware encoder
   ->
Bottleneck (embedding)
   ->
Decoder (for training only)


| TS2Vec               | FEMBA               |
| -------------------- | ------------------- |
| general time-series  | EEG-aware           |
| contrastive          | autoencoder         |
| temporal consistency | frequency structure |
| feature-agnostic     | band-aware          |

TS2Vec = baseline general
FEMBA = domain-specific

In [68]:
if MODE == 'colab':
  %pip install torch

In [69]:
import torch
import torch.nn as nn

In [70]:
# ================= Padding (same as TS2Vec) =================
max_len = max(seq.shape[0] for seq in sequences)
padded_sequences = np.array([
    np.pad(seq, ((0, max_len - seq.shape[0]), (0, 0)), mode='constant')
    for seq in sequences
])

X = torch.tensor(padded_sequences, dtype=torch.float32)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
X = X.to(device)

# ================= FEMBA Model =================
class FEMBA(nn.Module):
    def __init__(self, input_dim, hidden_dim=64, emb_dim=128):
        super().__init__()
        self.encoder_rnn = nn.GRU(input_dim, hidden_dim, batch_first=True)
        self.embedding = nn.Linear(hidden_dim, emb_dim)

        self.decoder_rnn = nn.GRU(emb_dim, hidden_dim, batch_first=True)
        self.output_layer = nn.Linear(hidden_dim, input_dim)

    def forward(self, x):
        _, h = self.encoder_rnn(x)
        z = self.embedding(h[-1])

        z_rep = z.unsqueeze(1).repeat(1, x.size(1), 1)
        dec_out, _ = self.decoder_rnn(z_rep)
        x_hat = self.output_layer(dec_out)
        return x_hat, z


model = FEMBA(
    input_dim=X.shape[2],
    hidden_dim=64,
    emb_dim=128
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.MSELoss()

# ================= Training =================
start = time.perf_counter()
print(">> FEMBA training starts <<")

epochs = 30
batch_size = 32

model.train()
for epoch in range(epochs):
    perm = torch.randperm(X.size(0))
    for i in range(0, X.size(0), batch_size):
        idx = perm[i:i+batch_size]
        batch = X[idx]

        optimizer.zero_grad()
        x_hat, _ = model(batch)
        loss = criterion(x_hat, batch)
        loss.backward()
        optimizer.step()

    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1}/{epochs} - loss: {loss.item():.4f}")

print(">> FEMBA training finished <<")
fit_time = time.perf_counter() - start


>> FEMBA training starts <<
Epoch 5/30 - loss: 15.8360
Epoch 10/30 - loss: 5.8097
Epoch 15/30 - loss: 10.5570
Epoch 20/30 - loss: 3.8758
Epoch 25/30 - loss: 3.7680
Epoch 30/30 - loss: 5.6396
>> FEMBA training finished <<


“The FEMBA model was trained until convergence of a reasonable reconstruction error, without over-optimizing the decoder, as the downstream task relies on the learned embeddings rather than perfect signal reconstruction.”

In [71]:
# ================= Embedding =================
start = time.perf_counter()
print(">> FEMBA embedding starts <<")

model.eval()
with torch.no_grad():
    _, embeddings = model(X)

embeddings = embeddings.cpu().numpy()  # (num_series, emb_dim)

print(">> FEMBA embedding finished <<")
encode_time = time.perf_counter() - start
print("embeddings shape:", embeddings.shape)

# ================= DataFrame =================
bin_femba_emb_df = pd.DataFrame(
    embeddings,
    columns=[f"femba_{i}" for i in range(embeddings.shape[1])]
)

bin_femba_emb_df["ad_break_session_key"] = keys

# time report
print("=" * 40)
print("FEMBA Feature Extraction Time Report")
print(f"FEMBA training time: {fit_time:.2f} seconds")
print(f"FEMBA encoding time: {encode_time:.2f} seconds")

print("=" * 40)
print("preview of embedding:")
bin_femba_emb_df.head()


>> FEMBA embedding starts <<
>> FEMBA embedding finished <<
embeddings shape: (2835, 128)
FEMBA Feature Extraction Time Report
FEMBA training time: 30.58 seconds
FEMBA encoding time: 0.22 seconds
preview of embedding:


,femba_0,femba_1,femba_2,femba_3,femba_4,femba_5,femba_6,femba_7,femba_8,femba_9,...,femba_119,femba_120,femba_121,femba_122,femba_123,femba_124,femba_125,femba_126,femba_127,ad_break_session_key
0,-0.606401,-1.106745,0.442651,-0.663982,0.168627,-0.493088,-0.325263,0.418764,0.192192,0.401326,...,0.060101,-0.508312,-0.642538,0.752084,0.196113,0.268139,0.464242,0.794021,0.350070,E:\AssoTV\sensor-data\O1_raw_20240607\< RemovedDueToNDA >_line...
1,0.426528,-0.630948,1.071725,-0.662809,-0.069342,0.223889,-0.700603,0.261015,0.900439,0.410420,...,0.056044,-0.365575,-0.146222,0.552068,0.189130,-0.035434,0.370090,-0.212572,-0.454786,E:\AssoTV\sensor-data\O1_raw_20240607\< RemovedDueToNDA >_line...
2,0.108897,-0.368999,0.762761,0.072321,0.439664,-0.492548,-0.862922,0.122503,-0.572019,0.900090,...,0.272193,-0.706183,-0.276656,0.340341,-0.284188,-1.177952,0.884600,0.196121,-0.278275,E:\AssoTV\sensor-data\O1_raw_20240607\< RemovedDueToNDA >_line...
3,0.173070,-0.943248,1.005233,0.493100,0.470159,-1.201443,0.517056,0.037774,-0.957127,1.331611,...,0.970998,-0.761423,0.781044,0.598314,0.864315,0.121346,0.470062,0.774856,0.476212,E:\AssoTV\sensor-data\O1_raw_20240607\< RemovedDueToNDA >_line...
4,0.468019,0.821087,0.629476,0.325954,-0.861197,0.124615,0.245031,0.082218,-0.338996,0.572879,...,0.099261,-0.604474,-0.218346,0.078229,0.974698,1.190120,-0.742797,-0.353990,-0.130992,E:\AssoTV\sensor-data\O1_raw_20240607\< RemovedDueToNDA >_line...


In [72]:
# Check for missing values
bin_femba_emb_df.isna().any().sum()


np.int64(0)

“FEMBA learns a compact latent representation by reconstructing frequency-aware EEG feature sequences through a bottleneck architecture.”

#### 3.3/ Simple Average of Bins (benchmark)

In [73]:
final_bin[feature_cols]

,z-a_mean-psd_Delta_frontal-midline_v^2/Hz,z-a_mean-psd_Delta_prefrontal_v^2/Hz,z-a_mean-psd_Delta_anterior-frontal_v^2/Hz,z-a_mean-psd_Delta_frontolateral_v^2/Hz,z-a_mean-psd_Delta_frontolateral-ext_v^2/Hz,z-a_mean-psd_Delta_left-frontolateral-ext_v^2/Hz,z-a_mean-psd_Delta_right-frontolateral-ext_v^2/Hz,z-a_mean-psd_Theta_frontal-midline_v^2/Hz,z-a_mean-psd_Theta_prefrontal_v^2/Hz,z-a_mean-psd_Theta_anterior-frontal_v^2/Hz,...,z-a_EI-2c_anterior-frontal,z-a_AI-1a_anterior-frontal,z-a_EI-2c_frontolateral,z-a_AI-1a_frontolateral,z-a_EI-2c_frontolateral-ext,z-a_AI-1a_frontolateral-ext,z-a_EI-2c_left-frontolateral-ext,z-a_AI-1a_left-frontolateral-ext,z-a_EI-2c_right-frontolateral-ext,z-a_AI-1a_right-frontolateral-ext
103,-0.227415,-0.301244,-0.017770,-0.677555,-0.352858,-0.538400,0.002715,0.252917,0.543335,-0.294490,...,-0.171257,0.449790,3.506366,-3.420137,0.841749,-1.486648,1.952629,-1.211509,-0.407400,-0.735477
104,-0.062315,-0.287198,0.544370,-2.848148,-1.136478,-2.408685,0.819986,0.456364,0.727825,-0.082306,...,-0.548759,0.053778,1.024614,-0.546652,-0.130428,-0.037726,-0.267956,-2.173891,-0.360511,2.513054
105,-0.102978,-0.260352,0.324589,-3.566601,-1.623194,-1.528726,-1.127486,1.923339,2.105500,1.363404,...,-1.165844,1.493060,4.228803,-5.047887,-0.123337,-1.822825,0.489738,-1.732282,-0.438168,-1.321724
106,7.051734,6.703275,7.630053,3.050094,5.817744,9.060820,-0.265967,3.231168,2.594604,3.966425,...,-1.676616,5.103680,2.700446,-1.048416,-0.992042,2.912018,1.206584,2.722615,-1.840417,-0.857547
107,-1.358885,-1.420164,-1.125660,2.182232,0.473844,1.097873,-0.454435,0.416594,-0.488733,1.975146,...,-1.237744,1.757839,-0.298199,2.720085,-1.100467,2.692055,-0.958782,3.255384,-0.384287,-0.975234
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
57338,1.090834,1.172644,1.180187,6.835048,5.198101,9.350225,0.874008,1.270845,1.215649,1.033885,...,10.152275,6.976653,-4.291989,12.259891,-4.343774,21.438581,-4.447681,31.094161,-1.895149,4.169803
57339,0.058402,0.260495,0.167041,1.706230,0.913098,1.471136,0.888652,0.762403,0.337936,1.002869,...,23.110435,9.878268,-3.793427,6.124263,-3.057829,9.388266,-3.982970,12.020511,3.473436,7.159921
57340,-0.666964,-0.651655,-0.696270,-1.745791,-1.119618,-2.038303,-0.163285,-0.603605,-0.539778,-0.732982,...,2.254350,-0.989859,2.395511,-1.411455,2.295826,-1.313599,3.309027,-1.909448,0.417694,0.892305
57341,1.316746,1.230135,1.502198,1.159917,1.399085,1.013342,1.776011,-0.026409,-0.508182,1.025162,...,-0.781307,-0.259241,4.705575,-1.034948,0.601440,-0.807956,2.575466,-1.639980,-1.768133,1.212980


In [74]:
%%time

print(">> simple means calculation starts <<")
start = time.perf_counter()
final_bin.sort_values(by=['ad_break_session_key', 'bin_order']).groupby('ad_break_session_key')[feature_cols].mean().reset_index()
encode_time = time.perf_counter() - start
print(">> simple means calculation finished <<")

# time report
print("=" * 40)
print("Simple Means Feature Extraction Time Report")
print(f"Simple means calculation time: {encode_time:.4f} seconds")

print("=" * 40)
print("preview of simple means calculation:")
final_bin.sort_values(by=['ad_break_session_key', 'bin_order']).groupby('ad_break_session_key')[feature_cols].mean().reset_index().head()

>> simple means calculation starts <<
>> simple means calculation finished <<
Simple Means Feature Extraction Time Report
Simple means calculation time: 0.0649 seconds
preview of simple means calculation:
CPU times: user 84.1 ms, sys: 906 µs, total: 85 ms
Wall time: 110 ms


,ad_break_session_key,z-a_mean-psd_Delta_frontal-midline_v^2/Hz,z-a_mean-psd_Delta_prefrontal_v^2/Hz,z-a_mean-psd_Delta_anterior-frontal_v^2/Hz,z-a_mean-psd_Delta_frontolateral_v^2/Hz,z-a_mean-psd_Delta_frontolateral-ext_v^2/Hz,z-a_mean-psd_Delta_left-frontolateral-ext_v^2/Hz,z-a_mean-psd_Delta_right-frontolateral-ext_v^2/Hz,z-a_mean-psd_Theta_frontal-midline_v^2/Hz,z-a_mean-psd_Theta_prefrontal_v^2/Hz,...,z-a_EI-2c_anterior-frontal,z-a_AI-1a_anterior-frontal,z-a_EI-2c_frontolateral,z-a_AI-1a_frontolateral,z-a_EI-2c_frontolateral-ext,z-a_AI-1a_frontolateral-ext,z-a_EI-2c_left-frontolateral-ext,z-a_AI-1a_left-frontolateral-ext,z-a_EI-2c_right-frontolateral-ext,z-a_AI-1a_right-frontolateral-ext
0,E:\AssoTV\sensor-data\O1_raw_20240607\< RemovedDueToNDA >_line...,0.872592,0.657668,1.405240,0.107920,0.841790,1.617468,-0.406968,1.544771,1.257347,...,-1.045259,2.159546,0.677983,0.552472,-0.818272,1.844158,-0.398148,2.557659,-0.649979,-0.608631
1,E:\AssoTV\sensor-data\O1_raw_20240607\< RemovedDueToNDA >_line...,-0.284831,-0.291646,-0.252129,-0.104566,-0.194155,-0.992384,0.838615,0.308452,0.026888,...,-0.405208,0.614013,-0.894356,1.892431,-0.625759,1.246576,-0.334940,0.538049,-0.613280,1.844283
2,E:\AssoTV\sensor-data\O1_raw_20240607\< RemovedDueToNDA >_line...,0.163217,0.063547,0.422450,-0.513565,-0.023266,-0.521361,0.584437,0.120488,0.455295,...,1.615269,-0.365131,2.070852,-0.997727,1.755723,-0.901634,1.361966,-0.707786,2.035861,-0.534901
3,E:\AssoTV\sensor-data\O1_raw_20240607\< RemovedDueToNDA >_line...,3.465125,3.371153,3.541965,5.510366,5.602515,3.219478,5.197306,-1.303942,-1.343799,...,1.495665,-1.014542,1.966603,-0.807836,1.574246,-0.901074,1.729659,-1.315443,1.430800,0.154220
4,E:\AssoTV\sensor-data\O1_raw_20240607\< RemovedDueToNDA >_line...,-0.943651,-0.912264,-0.980133,1.364176,1.699749,1.703905,1.056706,-1.197063,-1.255866,...,-0.311723,-0.700585,-1.367488,10.408203,-1.366697,17.911503,-0.895423,5.587913,-0.265522,32.024172


#### Export Femba Embedding DataFrame

In [75]:
# save the splits
if MODE == 'colab':
    femba_path = head_path + r'data/final_for_modeling/bin_femba_emb_df.parquet'

else:
    femba_path = r'../../data/final_for_modeling/bin_femba_emb_df.parquet'

bin_femba_emb_df.to_parquet(femba_path, index=False)